# Imports


In [2]:
import numpy as np
import torch
import time
import copy
import torch
import rendering
import heightfield
import image
import opt
import sys
import simannealing

In [3]:
print("Running with " + str(sys.version))

Running with 3.8.8 (default, Apr 13 2021, 12:59:45) 
[Clang 10.0.0 ]


# Grad descent

In [4]:
def grad_descent(num_epochs, H, COL, cam_directions, image_paths, use_barrier_loss = True, use_neighbor_loss = False, sim_annealing = True, optim_type = "Adam", heaviside_fn = None, heaviside_k = 0.1, num_repeats = 2, num_rays = 32):
  
  num_cams = len(cam_directions)
  num_strips = H.shape[0]*num_repeats
  ray_width = 1/num_rays
  
  original_ray_origins = [[], [], [], []]
  ray_origins = copy.deepcopy(original_ray_origins)
  coords = []
  
  schedule = torch.zeros(50) #update colors for 10 steps, then heights for 20 steps
  schedule[10:20] = 1
  schedule[40:50] = 1

  grad_types = schedule.repeat(20)
  

  for i in range(num_rays):
    for k in range(num_rays):
      coord = np.array([i*ray_width + ray_width/2, k*ray_width + ray_width/2, 1])
      coords.append(coord)

      for q in range(num_cams):   
        curr_origin = coord - 20*cam_directions[q]
        curr_origin[2] = curr_origin[2].item()
        ray_origins[q].append(curr_origin)



  H = torch.tensor(H, dtype=torch.float, requires_grad = True)
  COL = torch.tensor(COL, dtype=torch.float, requires_grad = True)



  ray_packs = []


  for t in range(num_cams):
    ray_packs.append(rendering.get_ray_packs(ray_origins[t], cam_directions[t], num_strips))

  
  desired_cols = []
  for i in range(num_cams):
    desired_cols.append(image.get_tensor_from_image(image_paths[i], (64, 64), (num_rays, num_rays)))
  
  
  H, COL = simannealing.sim_annealing(H, COL, ray_packs, num_cams, num_repeats, heaviside_fn, heaviside_k, desired_cols)
      
  if optim_type == "Adam":
    optimizer = torch.optim.AdamW([H, COL],lr=0.1)
    
  if optim_type == "SGD":
    optimizer = torch.optim.SGD([H, COL],lr=0.1, momentum=0.9)


  for j in range(1, num_epochs):
    
    torch.save(H, 'march20H.pt')
    torch.save(COL, 'march20COL.pt')
      
    change = False
    
    if j%100 == 0:
      H = heightfield.double_resolution(H)
      COL = heightfield.double_resolution(COL)
      H = torch.tensor(H, dtype=torch.float)
      COL = torch.tensor(COL, dtype=torch.float)
      num_strips = num_strips*2
      
      desired_cols = []
      for q in range(num_cams):

        desired_cols.append(image.get_tensor_from_image(image_paths[q], (64, 64), (num_rays, num_rays)))
      change = True


    if j%100 == 95:
      num_rays = num_rays*2
      ray_width = ray_width/2
      change = True
    
    if j%100 == 95 or j%100 == 0:
      coords = []
      change = True
      

      
      ray_origins = copy.deepcopy(original_ray_origins)
      for i in range(num_rays):
        for k in range(num_rays):
          coord = np.array([i*ray_width + ray_width/2, k*ray_width + ray_width/2, 1])
          coords.append(coord)
          for q in range(num_cams):    
            curr_origin = coord - 20*cam_directions[q] 
            curr_origin[2] = curr_origin[2].item()
            ray_origins[q].append(curr_origin)


      
      ray_packs = []
      for t in range(num_cams):
        ray_packs.append(rendering.get_ray_packs(ray_origins[t], cam_directions[t], num_strips))
        
      H.requires_grad = True
      COL.requires_grad = True
      
      desired_cols = []
      for i in range(num_cams):
        desired_cols.append(image.get_tensor_from_image(image_paths[i], (64, 64), (num_rays, num_rays)))

    if int(grad_types[j].item()) == 0:
      
      H.requires_grad = False
      COL.requires_grad = True
      

      lr = 0.05
      

    elif int(grad_types[j].item()) == 1:
      H.requires_grad = True
      COL.requires_grad = False 
      COL = COL.detach().numpy()
      COL = torch.tensor(COL, dtype=torch.float, requires_grad = False)
      if j > 49:
        lr = 0.03
      
    elif int(grad_types[j].item()) == 2:
      H.requires_grad = True
      COL.requires_grad = True
      H = torch.tensor(H, dtype=torch.float, requires_grad = True)
      COL = torch.tensor(COL, dtype=torch.float, requires_grad = True)

    if j > 0:
      if int(grad_types[j].item()) != int(grad_types[j-1].item()):
        
        change = True
    if change:



      if optim_type == "Adam":
        optimizer = torch.optim.AdamW([H, COL],lr=lr)
      if optim_type == "SGD":
        optimizer = torch.optim.SGD([H, COL],lr=lr, momentum=0.9)




    result_cols = rendering.forward(H, COL, ray_packs, num_cams, num_repeats=num_repeats, heaviside_fn = heaviside_fn, heaviside_k = heaviside_k)

    mse_loss = 0
    for q in range(num_cams):
      mse_loss += torch.nn.functional.mse_loss(result_cols[q].double(), desired_cols[q].double())




    if j % 1 == 0 or j == 99:
      print("EPOCH: " + str(j) + "\t LOSS: " + str(mse_loss.item()/2))
      time.sleep(1)
    if use_barrier_loss:
      barrier_loss = opt.barrier_fn(H, min_val = 0.1, max_val = 1.5)
      mse_loss = mse_loss + 0.02*barrier_loss

    if use_neighbor_loss:
      neighbor_loss = opt.neighbor_fn(H)
      mse_loss = mse_loss + 0.5*neighbor_loss



      
       
    
    optimizer.zero_grad()
    mse_loss.backward()
    optimizer.step()



  repeated_H = H.repeat(num_repeats, num_repeats)
  repeated_COL = COL.repeat(num_repeats, num_repeats, 1)


  return repeated_H, repeated_COL, result_cols, desired_cols



# Main

In [6]:
#azimuth compatibility ablation

H = heightfield.get_uniform_H(4, 4, center =0.85)
COL = heightfield.get_white_COL(4)

cam_direction_1 = np.array([1, 0.000001, -1])

cos = [1, 0.94, 0.766, 0.5, 0.174, -0.174, -0.5, -0.766, -0.94, -1]
sin = [0.001, 0.34, 0.643, 0.866, 0.984, 0.984, 0.866, 0.643, 0.34, 0.001]

np.set_printoptions(threshold=1000)
torch.set_printoptions(threshold = 1000)

cam_direction_1 = np.array([1, 0.0001, -1])
cam_direction_1 = np.array([1, 0.0001, -10])
cam_direction_3 = np.array([-1, 0.00001, -1])

cam_directions = np.array([cam_direction_1, cam_direction_2, cam_direction_3])

num_epochs = 260
p1 = "test_images/green.png"
p2 = "test_images/red.png"
p3 = "test_images/blue.png"
image_paths = [p1, p2, p3]
H1, COL1, result_cols, desired_cols = grad_descent(num_epochs, H, COL, cam_directions, image_paths, use_barrier_loss = True, use_neighbor_loss = False, optim_type="Adam", heaviside_fn = "tan", num_rays = 16)




<ipython-input-4-1e795c81fe92>:30: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  H = torch.tensor(H, dtype=torch.float, requires_grad = True)
<ipython-input-4-1e795c81fe92>:31: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  COL = torch.tensor(COL, dtype=torch.float, requires_grad = True)


STEP: 0	 LOSS: 1.7615577667397524
STEP: 1	 LOSS: 1.7468616992626356
STEP: 2	 LOSS: 1.7394374046898844
STEP: 3	 LOSS: 1.7353725486139373
STEP: 4	 LOSS: 1.7203449714660493
STEP: 5	 LOSS: 1.7129047592560953
STEP: 6	 LOSS: 1.690049208941796
STEP: 7	 LOSS: 1.6729059006104414
STEP: 8	 LOSS: 1.6722223762352764
STEP: 9	 LOSS: 1.658485249741613
STEP: 10	 LOSS: 1.6656156642777846
STEP: 11	 LOSS: 1.651717849140928
STEP: 12	 LOSS: 1.6469430229738937
STEP: 13	 LOSS: 1.6567501594514105
STEP: 14	 LOSS: 1.6471857882203298
STEP: 15	 LOSS: 1.6495132026061736
STEP: 16	 LOSS: 1.6457877464313968
STEP: 17	 LOSS: 1.6403889623471615
STEP: 18	 LOSS: 1.6399936919522307
STEP: 19	 LOSS: 1.6438029424120861
STEP: 20	 LOSS: 1.6394576435745882
STEP: 21	 LOSS: 1.6293114564452102
STEP: 22	 LOSS: 1.6244256303263507
STEP: 23	 LOSS: 1.6266798805450928
STEP: 24	 LOSS: 1.623337558102921
STEP: 25	 LOSS: 1.6175657769672278
STEP: 26	 LOSS: 1.619005191778524
STEP: 27	 LOSS: 1.6199119051124682
STEP: 28	 LOSS: 1.6044930319770705


<ipython-input-4-1e795c81fe92>:66: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  H = torch.tensor(H, dtype=torch.float)
<ipython-input-4-1e795c81fe92>:67: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  COL = torch.tensor(COL, dtype=torch.float)


EPOCH: 100	 LOSS: 0.19313138848739586
EPOCH: 101	 LOSS: 0.17825019932945013
EPOCH: 102	 LOSS: 0.16720587799593808
EPOCH: 103	 LOSS: 0.16013417532413238
EPOCH: 104	 LOSS: 0.15566068311826345
EPOCH: 105	 LOSS: 0.15243593144659462
EPOCH: 106	 LOSS: 0.15018561176471007
EPOCH: 107	 LOSS: 0.14896663554122117
EPOCH: 108	 LOSS: 0.14863353760971185
EPOCH: 109	 LOSS: 0.14882037542752796
EPOCH: 110	 LOSS: 0.1492288776746239
EPOCH: 111	 LOSS: 0.148471188923552
EPOCH: 112	 LOSS: 0.14934424642192942
EPOCH: 113	 LOSS: 0.14488904737360195
EPOCH: 114	 LOSS: 0.14444827622266432
EPOCH: 115	 LOSS: 0.1465656180074252
EPOCH: 116	 LOSS: 0.14755683941070583
EPOCH: 117	 LOSS: 0.14734034839392002
EPOCH: 118	 LOSS: 0.1473634993228322
EPOCH: 119	 LOSS: 0.1486912625843221
EPOCH: 120	 LOSS: 0.149460817921465
EPOCH: 121	 LOSS: 0.14187083611246115
EPOCH: 122	 LOSS: 0.13599440050583242
EPOCH: 123	 LOSS: 0.13296260711446642
EPOCH: 124	 LOSS: 0.13116172669728607
EPOCH: 125	 LOSS: 0.12951877942638318
EPOCH: 126	 LOSS: 0.